# Training Chronos-2-Small from Scratch

This notebook trains a `chronos-2-small` (~28M params) model from scratch using:
- **Real data**: M4 daily, hourly, and monthly datasets from `autogluon/chronos_datasets`
- **Synthetic data**: KernelSynth GP-based synthetic time series

Training is done in two stages:
1. **Stage 1**: `context_length=2048`, `prediction_length=256` for 180k steps
2. **Stage 2**: `context_length=8192`, `prediction_length=1024` for 20k steps

After training, we evaluate in-domain on the same 3 M4 datasets using MASE and WQL metrics.

## Cell 1: Setup & Dependencies

In [ ]:
%pip install -e "..[extras,dev]" wandb

In [ ]:
import os
import math
import random
import functools
from pathlib import Path

import numpy as np
import torch
import transformers
from transformers import TrainingArguments
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    ConstantKernel,
    DotProduct,
    ExpSineSquared,
    Kernel,
    RationalQuadratic,
    WhiteKernel,
)
from tqdm.auto import tqdm
import datasets as hf_datasets

from chronos.chronos2.config import Chronos2CoreConfig, Chronos2ForecastingConfig
from chronos.chronos2.model import Chronos2Model
from chronos.chronos2.dataset import Chronos2Dataset, DatasetMode
from chronos.chronos2.trainer import Chronos2Trainer
from chronos.chronos2.pipeline import Chronos2Pipeline

# Reproducibility
torch.manual_seed(42)
transformers.set_seed(42)
np.random.seed(42)
random.seed(42)

os.environ["WANDB_PROJECT"] = "chronos-2-small-training"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

## Cell 2: Configuration

In [ ]:
# --- Model architecture (chronos-2-small) ---
D_MODEL = 512
D_KV = 64
D_FF = 2048
NUM_LAYERS = 6
NUM_HEADS = 8
DROPOUT_RATE = 0.1
INITIALIZER_FACTOR = 0.05
VOCAB_SIZE = 2

# --- Chronos forecasting config ---
INPUT_PATCH_SIZE = 16
INPUT_PATCH_STRIDE = 16
OUTPUT_PATCH_SIZE = 16
QUANTILES = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
USE_ARCSINH = True
USE_REG_TOKEN = True

# --- Stage 1 training ---
STAGE1_CONTEXT_LENGTH = 2048
STAGE1_MAX_OUTPUT_PATCHES = 16  # prediction_length = 16 * 16 = 256
STAGE1_PREDICTION_LENGTH = STAGE1_MAX_OUTPUT_PATCHES * OUTPUT_PATCH_SIZE
STAGE1_MAX_STEPS = 180_000
STAGE1_TIME_ENCODING_SCALE = STAGE1_CONTEXT_LENGTH

# --- Stage 2 training (long-context) ---
STAGE2_CONTEXT_LENGTH = 8192
STAGE2_MAX_OUTPUT_PATCHES = 64  # prediction_length = 64 * 16 = 1024
STAGE2_PREDICTION_LENGTH = STAGE2_MAX_OUTPUT_PATCHES * OUTPUT_PATCH_SIZE
STAGE2_MAX_STEPS = 20_000
STAGE2_TIME_ENCODING_SCALE = STAGE2_CONTEXT_LENGTH

# --- Optimizer ---
LEARNING_RATE = 1e-3
LR_SCHEDULER = "linear"
WARMUP_RATIO = 0.0
WEIGHT_DECAY = 0.01
OPTIM = "adamw_torch_fused"

# --- Batch / hardware ---
BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
SAVE_STEPS = 50_000
LOGGING_STEPS = 500

# --- Data generation ---
NUM_KERNELSYNTH_SERIES = 100_000
KERNELSYNTH_LENGTH = 2048  # long enough for Stage 2 (needs >= min_past + prediction_length)

# --- Output ---
OUTPUT_DIR = Path("./output/chronos2-small-pretrain")

# --- Evaluation datasets ---
DATASET_CONFIGS = {
    "m4_daily":   {"prediction_length": 14,  "offset": -14},
    "m4_hourly":  {"prediction_length": 48,  "offset": -48},
    "m4_monthly": {"prediction_length": 18,  "offset": -18},
}

print(f"Stage 1: context={STAGE1_CONTEXT_LENGTH}, pred_len={STAGE1_PREDICTION_LENGTH}, steps={STAGE1_MAX_STEPS}")
print(f"Stage 2: context={STAGE2_CONTEXT_LENGTH}, pred_len={STAGE2_PREDICTION_LENGTH}, steps={STAGE2_MAX_STEPS}")

## Cell 3: Data Preparation

### 3a: Load M4 datasets

In [ ]:
real_train_data = []
eval_data = {}  # kept separate for in-domain evaluation

for ds_name, ds_cfg in DATASET_CONFIGS.items():
    ds = hf_datasets.load_dataset(
        "autogluon/chronos_datasets", ds_name, split="train", trust_remote_code=True
    )
    offset = ds_cfg["offset"]
    test_series = []
    for row in ds:
        target = np.array(row["target"], dtype=np.float32)
        # Training: everything up to offset (exclude last prediction_length values)
        real_train_data.append({"target": target[:offset]})
        # Eval: store full series
        test_series.append({"target": target})
    eval_data[ds_name] = test_series
    print(f"  {ds_name}: {len(test_series)} series")

print(f"\nTotal real training series: {len(real_train_data)}")

### 3b: Generate KernelSynth synthetic data

In [ ]:
# KernelSynth kernel bank (from scripts/kernel-synth.py)
KERNEL_BANK = [
    ExpSineSquared(periodicity=24 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=48 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=96 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=24 * 7 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=48 * 7 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=96 * 7 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=7 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=14 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=30 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=60 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=365 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=365 * 2 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=4 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=26 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=52 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=4 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=6 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=12 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=4 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=4 * 10 / KERNELSYNTH_LENGTH),
    ExpSineSquared(periodicity=10 / KERNELSYNTH_LENGTH),
    DotProduct(sigma_0=0.0),
    DotProduct(sigma_0=1.0),
    DotProduct(sigma_0=10.0),
    RBF(length_scale=0.1),
    RBF(length_scale=1.0),
    RBF(length_scale=10.0),
    RationalQuadratic(alpha=0.1),
    RationalQuadratic(alpha=1.0),
    RationalQuadratic(alpha=10.0),
    WhiteKernel(noise_level=0.1),
    WhiteKernel(noise_level=1.0),
    ConstantKernel(),
]


def random_binary_map(a: Kernel, b: Kernel):
    binary_maps = [lambda x, y: x + y, lambda x, y: x * y]
    return np.random.choice(binary_maps)(a, b)


def sample_from_gp_prior(kernel: Kernel, X: np.ndarray):
    if X.ndim == 1:
        X = X[:, None]
    gpr = GaussianProcessRegressor(kernel=kernel)
    return gpr.sample_y(X, n_samples=1, random_state=None)


def generate_time_series(length: int = KERNELSYNTH_LENGTH, max_kernels: int = 5):
    while True:
        X = np.linspace(0, 1, length)
        selected_kernels = np.random.choice(
            KERNEL_BANK, np.random.randint(1, max_kernels + 1), replace=True
        )
        kernel = functools.reduce(random_binary_map, selected_kernels)
        try:
            ts = sample_from_gp_prior(kernel=kernel, X=X)
        except np.linalg.LinAlgError:
            continue
        return {"target": ts.squeeze().astype(np.float32)}


print(f"Generating {NUM_KERNELSYNTH_SERIES} synthetic series (length={KERNELSYNTH_LENGTH})...")
synthetic_data = [generate_time_series() for _ in tqdm(range(NUM_KERNELSYNTH_SERIES))]
print(f"Generated {len(synthetic_data)} synthetic series")

### 3c: Combine all training data

In [ ]:
all_training_data = real_train_data + synthetic_data
random.shuffle(all_training_data)
print(f"Total training series: {len(all_training_data)}")

## Cell 4: Model Initialization

In [ ]:
config = Chronos2CoreConfig(
    d_model=D_MODEL,
    d_kv=D_KV,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout_rate=DROPOUT_RATE,
    layer_norm_epsilon=1e-6,
    initializer_factor=INITIALIZER_FACTOR,
    feed_forward_proj="relu",
    vocab_size=VOCAB_SIZE,
    pad_token_id=0,
    rope_theta=10000.0,
)

config.chronos_config = {
    "context_length": STAGE1_CONTEXT_LENGTH,
    "input_patch_size": INPUT_PATCH_SIZE,
    "input_patch_stride": INPUT_PATCH_STRIDE,
    "output_patch_size": OUTPUT_PATCH_SIZE,
    "max_output_patches": STAGE1_MAX_OUTPUT_PATCHES,
    "quantiles": QUANTILES,
    "time_encoding_scale": STAGE1_TIME_ENCODING_SCALE,
    "use_arcsinh": USE_ARCSINH,
    "use_reg_token": USE_REG_TOKEN,
}
config.chronos_pipeline_class = "Chronos2Pipeline"
config.architectures = ["Chronos2Model"]
config.reg_token_id = 1

model = Chronos2Model(config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Cell 5: Stage 1 — Pretraining (context_length=2048)

In [ ]:
# Create Stage 1 dataset
train_dataset_s1 = Chronos2Dataset(
    inputs=all_training_data,
    context_length=STAGE1_CONTEXT_LENGTH,
    prediction_length=STAGE1_PREDICTION_LENGTH,
    batch_size=BATCH_SIZE,
    output_patch_size=OUTPUT_PATCH_SIZE,
    min_past=64,
    mode=DatasetMode.TRAIN,
    convert_inputs=True,
)
print(f"Stage 1 dataset: {len(train_dataset_s1.inputs)} series after filtering")

In [ ]:
stage1_output_dir = OUTPUT_DIR / "stage1"

training_args_s1 = TrainingArguments(
    output_dir=str(stage1_output_dir),
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    optim=OPTIM,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    report_to=["wandb"],
    max_steps=STAGE1_MAX_STEPS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    dataloader_num_workers=0,
    tf32=True,
    torch_compile=True,
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
    save_only_model=True,
    bf16=True,
)

trainer_s1 = Chronos2Trainer(
    model=model,
    args=training_args_s1,
    train_dataset=train_dataset_s1,
)

trainer_s1.train()

# Save Stage 1 checkpoint
stage1_ckpt = stage1_output_dir / "checkpoint-stage1-final"
model.save_pretrained(stage1_ckpt)
print(f"Stage 1 model saved to {stage1_ckpt}")

## Cell 6: Stage 2 — Long-Context Post-Training (context_length=8192)

In [ ]:
# Update model config for Stage 2
model.chronos_config.context_length = STAGE2_CONTEXT_LENGTH
model.chronos_config.max_output_patches = STAGE2_MAX_OUTPUT_PATCHES
model.chronos_config.time_encoding_scale = STAGE2_TIME_ENCODING_SCALE
model.config.chronos_config["context_length"] = STAGE2_CONTEXT_LENGTH
model.config.chronos_config["max_output_patches"] = STAGE2_MAX_OUTPUT_PATCHES
model.config.chronos_config["time_encoding_scale"] = STAGE2_TIME_ENCODING_SCALE

# Create Stage 2 dataset
train_dataset_s2 = Chronos2Dataset(
    inputs=all_training_data,
    context_length=STAGE2_CONTEXT_LENGTH,
    prediction_length=STAGE2_PREDICTION_LENGTH,
    batch_size=BATCH_SIZE,
    output_patch_size=OUTPUT_PATCH_SIZE,
    min_past=64,
    mode=DatasetMode.TRAIN,
    convert_inputs=True,
)
print(f"Stage 2 dataset: {len(train_dataset_s2.inputs)} series after filtering")

In [ ]:
stage2_output_dir = OUTPUT_DIR / "stage2"

training_args_s2 = TrainingArguments(
    output_dir=str(stage2_output_dir),
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    optim=OPTIM,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    report_to=["wandb"],
    max_steps=STAGE2_MAX_STEPS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    dataloader_num_workers=0,
    tf32=True,
    torch_compile=True,
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
    save_only_model=True,
    bf16=True,
)

trainer_s2 = Chronos2Trainer(
    model=model,
    args=training_args_s2,
    train_dataset=train_dataset_s2,
)

trainer_s2.train()

# Save final checkpoint
final_ckpt = OUTPUT_DIR / "checkpoint-final"
model.config.chronos_config = {
    k: v for k, v in model.chronos_config.__dict__.items()
}
model.save_pretrained(final_ckpt)
print(f"Final model saved to {final_ckpt}")

## Cell 7: In-Domain Evaluation on M4 Datasets

Evaluate the trained model on `m4_daily`, `m4_hourly`, and `m4_monthly` using MASE and WQL (Weighted Quantile Loss).

In [ ]:
import wandb

pipeline = Chronos2Pipeline.from_pretrained(str(final_ckpt), device_map="cuda")

EVAL_QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
MEDIAN_IDX = EVAL_QUANTILE_LEVELS.index(0.5)  # index 4

results = {}

for ds_name, ds_cfg in DATASET_CONFIGS.items():
    pred_len = ds_cfg["prediction_length"]
    offset = ds_cfg["offset"]

    # Prepare inputs: context = series[:offset], ground_truth = series[offset:]
    contexts = [
        {"target": torch.tensor(s["target"][:offset], dtype=torch.float32)}
        for s in eval_data[ds_name]
    ]
    ground_truths = [
        np.array(s["target"][offset:], dtype=np.float32)
        for s in eval_data[ds_name]
    ]

    # Get quantile predictions
    quantiles_list, mean_list = pipeline.predict_quantiles(
        contexts, prediction_length=pred_len, quantile_levels=EVAL_QUANTILE_LEVELS
    )

    # Compute MASE and WQL per series, then average
    mase_vals = []
    wql_vals = []

    for i, (q_pred, gt) in enumerate(zip(quantiles_list, ground_truths)):
        # q_pred shape: (n_variates, prediction_length, num_quantiles)
        median = q_pred[0, :, MEDIAN_IDX].cpu().numpy()
        full_target = np.array(eval_data[ds_name][i]["target"], dtype=np.float32)
        history = full_target[:offset]

        # MASE: mean absolute error / naive seasonal=1 scaling
        naive_scale = np.mean(np.abs(np.diff(history)))
        if naive_scale < 1e-9:
            naive_scale = 1.0
        mase = np.mean(np.abs(median - gt)) / naive_scale
        mase_vals.append(mase)

        # WQL: average pinball loss across quantiles, normalized by sum(|gt|)
        total_ql = 0.0
        for qi, q_level in enumerate(EVAL_QUANTILE_LEVELS):
            q_forecast = q_pred[0, :, qi].cpu().numpy()
            error = gt - q_forecast
            ql = np.where(error >= 0, q_level * error, (q_level - 1) * error)
            total_ql += np.sum(ql)
        abs_target_sum = np.sum(np.abs(gt))
        wql = total_ql / max(abs_target_sum, 1e-9) / len(EVAL_QUANTILE_LEVELS)
        wql_vals.append(wql)

    results[ds_name] = {"MASE": float(np.mean(mase_vals)), "WQL": float(np.mean(wql_vals))}
    print(f"{ds_name}: MASE={results[ds_name]['MASE']:.4f}, WQL={results[ds_name]['WQL']:.6f}")

# Log to W&B
for ds_name, metrics in results.items():
    wandb.log({
        f"eval/{ds_name}/MASE": metrics["MASE"],
        f"eval/{ds_name}/WQL": metrics["WQL"],
    })

wandb.finish()
print("\nAll evaluation results logged to W&B.")

## Cell 8: Save Final Model

In [ ]:
final_save_path = "./output/chronos2-small-final"
pipeline.model.save_pretrained(final_save_path)
print(f"Model saved to {final_save_path}")